# 06 - Full Tutorial: Voice AI Study Coach
This notebook is a step-by-step, end-to-end tutorial version of the project.

You will learn and run:
1. Configuration and local backend checks
2. Knowledge loading and retrieval
3. Voice I/O (procedural synthesis + sidecar transcript)
4. A full tutoring session orchestration
5. Batch demo runs, evaluation, telemetry, and report inspection

## Step 1 - Imports and Project Settings
Load all key modules used by the voice tutoring system.

In [1]:
from pathlib import Path
import csv
import json

from voice_study_coach.config import get_settings
from voice_study_coach.ollama_client import AsyncOllamaGateway
from voice_study_coach.tools.knowledge_base import load_docs, retrieve
from voice_study_coach.audio.synthesizer import ProceduralVoiceSynthesizer
from voice_study_coach.audio.transcriber import SidecarTranscriber
from voice_study_coach.agents.tutor import TutorAgent
from voice_study_coach.agents.quiz import QuizAgent
from voice_study_coach.orchestration.session import VoiceStudySession
from voice_study_coach.telemetry.tracer import JsonlTelemetryTracer, summarize_traces
from voice_study_coach.pipeline import run_demo, run_evaluation

settings = get_settings()
settings

Settings(project_root=PosixPath('/home/ahmad/AI/Projects/voice-ai-study-coach'), ollama_host='http://127.0.0.1:11434', tutor_model='phi3.5:3.8b', quiz_model='functiongemma:270m', seed=42, retrieval_top_k=3, generation_temperature=0.1, generation_max_tokens=180, generation_timeout_seconds=3.0, audio_sample_rate=16000, audio_amplitude=0.3, word_seconds=0.11, pause_seconds=0.03, knowledge_dir=PosixPath('data/knowledge'), evaluation_file=PosixPath('data/eval/tasks.json'), demo_audio_dir=PosixPath('data/demo_audio'), artifacts_dir=PosixPath('artifacts'), audio_dir=PosixPath('artifacts/audio'), eval_dir=PosixPath('artifacts/evals'), report_dir=PosixPath('artifacts/reports'), runs_dir=PosixPath('artifacts/runs'), telemetry_dir=PosixPath('artifacts/telemetry'))

## Step 2 - Runtime Configuration Snapshot
Print the practical runtime knobs that affect behavior, latency, and outputs.

In [2]:
config_view = {
    'ollama_host': settings.ollama_host,
    'tutor_model': settings.tutor_model,
    'quiz_model': settings.quiz_model,
    'retrieval_top_k': settings.retrieval_top_k,
    'generation_timeout_seconds': settings.generation_timeout_seconds,
    'audio_sample_rate': settings.audio_sample_rate,
    'audio_amplitude': settings.audio_amplitude,
}
config_view

{'ollama_host': 'http://127.0.0.1:11434',
 'tutor_model': 'phi3.5:3.8b',
 'quiz_model': 'functiongemma:270m',
 'retrieval_top_k': 3,
 'generation_timeout_seconds': 3.0,
 'audio_sample_rate': 16000,
 'audio_amplitude': 0.3}

## Step 3 - Check Local Model Availability
This confirms whether the configured Ollama models are locally available.

In [3]:
gateway = AsyncOllamaGateway(settings.ollama_host)
local_models = await gateway.list_model_names()
{
    'configured_models': [settings.tutor_model, settings.quiz_model],
    'available_subset': [m for m in [settings.tutor_model, settings.quiz_model] if m in local_models],
    'n_local_models': len(local_models),
}

{'configured_models': ['phi3.5:3.8b', 'functiongemma:270m'],
 'available_subset': ['phi3.5:3.8b', 'functiongemma:270m'],
 'n_local_models': 27}

## Step 4 - Load Knowledge Base Documents
The study coach retrieves from these local markdown files.

In [4]:
docs = load_docs(settings.resolved_knowledge_dir)
{
    'n_docs': len(docs),
    'sources': [doc.source for doc in docs],
}

{'n_docs': 5,
 'sources': ['data_policy.md',
  'incident_playbook.md',
  'release_notes_policy.md',
  'study_tips.md',
  'support_handbook.md']}

## Step 5 - Retrieval Demonstration
Run lexical retrieval for one question and inspect ranked sources.

In [5]:
sample_question = 'When is emergency rollback mandatory for API incidents?'
hits = retrieve(sample_question, docs, top_k=settings.retrieval_top_k)
[
    {'source': item.doc.source, 'score': round(item.score, 4)}
    for item in hits
]

[{'source': 'incident_playbook.md', 'score': 0.875},
 {'source': 'data_policy.md', 'score': 0.125},
 {'source': 'release_notes_policy.md', 'score': 0.125}]

## Step 6 - Build Voice Input Example
Create a WAV question and sidecar transcript (`.txt`) so the transcriber can decode it.

In [6]:
synth = ProceduralVoiceSynthesizer(
    sample_rate=settings.audio_sample_rate,
    amplitude=settings.audio_amplitude,
    word_seconds=settings.word_seconds,
    pause_seconds=settings.pause_seconds,
)
transcriber = SidecarTranscriber()

tutorial_audio = settings.resolved_demo_audio_dir / 'tutorial_student_question.wav'
tutorial_text = 'Can you teach me the rollback threshold and duration for API incidents?'
duration = synth.synthesize(tutorial_text, tutorial_audio)
tutorial_audio.with_suffix('.txt').write_text(tutorial_text, encoding='utf-8')

{'audio_path': tutorial_audio.as_posix(), 'duration_seconds': round(duration, 3)}

{'audio_path': '/home/ahmad/AI/Projects/voice-ai-study-coach/data/demo_audio/tutorial_student_question.wav',
 'duration_seconds': 1.746}

## Step 7 - Verify Transcription
Transcriber reads the sidecar transcript associated with the WAV input.

In [7]:
transcript_result = transcriber.transcribe(tutorial_audio)
transcript_result

TranscriptResult(transcript='Can you teach me the rollback threshold and duration for API incidents?', method='sidecar')

## Step 8 - Build and Run One Full Session Manually
Here we instantiate agents + orchestrator directly to make the flow explicit.

In [8]:
tutor = TutorAgent(settings=settings, gateway=gateway)
quiz = QuizAgent(settings=settings, gateway=gateway)
tracer = JsonlTelemetryTracer(settings.traces_file)
session = VoiceStudySession(
    tutor=tutor,
    quiz=quiz,
    transcriber=transcriber,
    synthesizer=synth,
    docs=docs,
    top_k=settings.retrieval_top_k,
    audio_output_dir=settings.resolved_audio_dir,
    tracer=tracer,
)

single_run = await session.run(trace_id='tutorial-single-audio', question_audio_path=tutorial_audio)
{
    'question_text': single_run.question_text,
    'top_retrieved': single_run.retrieved[:2],
    'tutor_audio_seconds': single_run.tutor_audio_seconds,
    'quiz_audio_seconds': single_run.quiz_audio_seconds,
    'tutor_fallback_used': single_run.tutor_fallback_used,
    'quiz_fallback_used': single_run.quiz_fallback_used,
}

{'question_text': 'Can you teach me the rollback threshold and duration for API incidents?',
 'top_retrieved': [{'source': 'incident_playbook.md',
   'title': 'Incident Playbook',
   'score': 0.3333,
   'text': '# Incident Playbook\n\nFor API incidents, emergency rollback is mandatory if **p95 latency exceeds 1200 ms for 10 consecutive minutes**.\n\nPrimary paging system is **Solar Pager**.\n\nSeverity levels: S0 outage, S1 critical degradation, S2 moderate degradation, S3 low impact.'},
  {'source': 'data_policy.md',
   'title': 'Data Policy',
   'score': 0.1667,
   'text': '# Data Policy\n\nRaw event logs are retained for **400 days**.\n\nFull backups run Sunday 02:30 UTC and are retained for 30 days.'}],
 'tutor_audio_seconds': 7.846,
 'quiz_audio_seconds': 1.774,
 'tutor_fallback_used': True,
 'quiz_fallback_used': True}

## Step 9 - Inspect Generated Audio Files
Confirm that tutor/quiz audio artifacts were produced by this session.

In [9]:
audio_files = sorted(settings.resolved_audio_dir.glob('tutorial-single-audio*.wav'))
[{'file': p.name, 'size_kb': round(p.stat().st_size / 1024, 2)} for p in audio_files]

[{'file': 'tutorial-single-audio_quiz.wav', 'size_kb': 55.46},
 {'file': 'tutorial-single-audio_tutor.wav', 'size_kb': 245.25}]

## Step 10 - Run Batch Demo Sessions (Pipeline Helper)
This executes the project's predefined demo sessions and stores them.

In [10]:
demo_runs = await run_demo(settings)
{
    'demo_count': len(demo_runs),
    'demo_runs_file': settings.demo_runs_file.as_posix(),
    'first_demo_trace_id': demo_runs[0]['trace_id'],
}

{'demo_count': 3,
 'demo_runs_file': '/home/ahmad/AI/Projects/voice-ai-study-coach/artifacts/runs/demo_sessions.json',
 'first_demo_trace_id': 'demo-audio-001'}

## Step 11 - Run Evaluation
Compare baseline vs tutor answer quality across the task set.

In [11]:
eval_payload = await run_evaluation(settings, reset_traces=False)
eval_payload['summary']

{'n_tasks': 6,
 'baseline_keyword_recall_mean': 0.0,
 'tutor_keyword_recall_mean': 1.0,
 'keyword_recall_gain': 1.0,
 'retrieval_hit_rate': 1.0,
 'source_citation_rate': 1.0,
 'tutor_fallback_rate': 1.0,
 'quiz_fallback_rate': 1.0,
 'avg_tutor_audio_seconds': 6.0231666666666674,
 'avg_total_latency_ms': 7026.853166666667}

## Step 12 - Inspect Prediction Rows
Read a few rows from the evaluation CSV artifact.

In [12]:
rows = []
with open(settings.predictions_file, encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    for idx, row in enumerate(reader):
        rows.append(row)
        if idx >= 2:
            break
rows

[{'task_id': 'q1',
  'question': 'When should emergency rollback be triggered for API incidents?',
  'expected_source': 'incident_playbook.md',
  'baseline_answer': 'I do not know from the question alone.',
  'tutor_answer': 'From incident_playbook.md: # Incident Playbook\n\nFor API incidents, emergency rollback is mandatory if **p95 latency exceeds 1200 ms for 10 consecutive minutes**.\n\nPrimary paging system is **Solar Pager**.\n\nSeverity levels: S0 outage, S1 critical degradation, S2 moderate degradation, S3 low impact.\nThis fallback summary was produced because live model generation timed out.',
  'baseline_keyword_recall': '0.0',
  'tutor_keyword_recall': '1.0',
  'keyword_gain': '1.0',
  'retrieval_hit': 'True',
  'source_cited': 'True',
  'tutor_fallback_used': 'True',
  'quiz_fallback_used': 'True',
  'tutor_audio_seconds': '7.846',
  'total_latency_ms': '7023.702'},
 {'task_id': 'q2',
  'question': 'What is the weekly release train schedule?',
  'expected_source': 'release_

## Step 13 - Telemetry Summary
Aggregate trace spans and examine latency by stage.

In [13]:
telemetry = summarize_traces(settings.traces_file, settings.telemetry_summary_file)
{
    'n_spans': telemetry['n_spans'],
    'n_unique_traces': telemetry['n_unique_traces'],
    'first_3_spans': telemetry['by_span'][:3],
}

{'n_spans': 175,
 'n_unique_traces': 10,
 'first_3_spans': [{'span_name': 'baseline_answer',
   'count': 28,
   'error_count': 0,
   'latency_ms_mean': 2007.853,
   'latency_ms_p50': 2003.874,
   'latency_ms_p95': 2015.986,
   'latency_ms_max': 2088.29},
  {'span_name': 'quiz_generation',
   'count': 28,
   'error_count': 0,
   'latency_ms_mean': 2004.269,
   'latency_ms_p50': 2003.87,
   'latency_ms_p95': 2008.116,
   'latency_ms_max': 2008.944},
  {'span_name': 'retrieve_context',
   'count': 28,
   'error_count': 0,
   'latency_ms_mean': 0.061,
   'latency_ms_p50': 0.059,
   'latency_ms_p95': 0.073,
   'latency_ms_max': 0.083}]}

## Step 14 - Report Preview
Load the generated markdown report and preview the first section.

In [14]:
report_text = Path(settings.report_file).read_text(encoding='utf-8')
print(report_text[:1800])

# Voice AI Study Coach Report

Generated at: `2026-06-12T11:44:32+00:00`

## Setup

- Tutor model: `phi3.5:3.8b`
- Quiz model: `functiongemma:270m`
- Tasks evaluated: **6**

## Metrics

- Baseline keyword recall mean: **0.0000**
- Tutor keyword recall mean: **1.0000**
- Keyword recall gain: **1.0000**
- Retrieval hit rate: **1.0000**
- Source citation rate: **1.0000**
- Tutor fallback rate: **1.0000**
- Quiz fallback rate: **1.0000**
- Avg tutor audio duration (s): **6.023**
- Avg total latency (ms): **7026.85**

## Telemetry

- Total spans: **175**
- Unique traces: **10**

| Span | Count | Mean ms | P95 ms | Errors |
|---|---:|---:|---:|---:|
| baseline_answer | 28 | 2007.853 | 2015.986 | 0 |
| quiz_generation | 28 | 2004.269 | 2008.116 | 0 |
| retrieve_context | 28 | 0.061 | 0.073 | 0 |
| synthesize_quiz_audio | 28 | 1.884 | 5.943 | 0 |
| synthesize_tutor_audio | 28 | 8.86 | 15.146 | 0 |
| transcribe_audio | 7 | 0.085 | 0.104 | 0 |
| tutor_answer | 28 | 3005.983 | 3012.842 | 0 |


##

## Tutorial Complete
You now have an end-to-end, executable walkthrough with real outputs saved to disk.